<a href="https://colab.research.google.com/github/EgemenYapucu/DataScienceExercises/blob/main/DataPreProcessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Mount our google drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip3 install face_recognition

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.1/100.1 MB 8.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for face-recognition-models: filename=face_recognition_models-0.3.0-py2.py3-none-any.whl size=100566173 sha256=2657bbfb516505b2012bc6d4870c856437e43f9bae6cb6c00f203e1c4cbdb2f2
  Stored in directory: /root/.cache/pip/wheels/7a/eb/cf/e9eced74122b679557f597bb7c8e4c739cfcac526db1fd523d
Successfully built face-recognition-models


In [ ]:
import json
import glob
import numpy as np
import cv2
import copy
import torch
import torchvision
from torchvision import transforms
from torch.utils.data import DataLoader
from torch.utils.data.dataset import Dataset
import os
import csv
import shutil
import matplotlib.pyplot as plt
import face_recognition
from tqdm.autonotebook import tqdm

In [ ]:
#To get the average frame count
video_files =  glob.glob('/content/drive/MyDrive/dfdc_train_part_1/*.mp4')
frame_count = []
for video_file in video_files:
  cap = cv2.VideoCapture(video_file)
  if(int(cap.get(cv2.CAP_PROP_FRAME_COUNT))<150):
    video_files.remove(video_file)
    continue
  frame_count.append(int(cap.get(cv2.CAP_PROP_FRAME_COUNT)))
print("frames" , frame_count)
print("Total number of videos: " , len(frame_count))
print('Average frame per video:',np.mean(frame_count))

<ipython-input-4-cc70130fbafb>:17: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


KeyboardInterrupt: ignored

In [ ]:
# to extract frame
def frame_extract(path):
  vidObj = cv2.VideoCapture(path)
  success = 1
  while success:
      success, image = vidObj.read()
      if success:
          yield image
!mkdir '/content/drive/My Drive/FaceData'
# process the frames
def create_face_videos(path_list,out_dir):
  already_present_count =  glob.glob(out_dir+'*.mp4')
  print("No of videos already present " , len(already_present_count))
  for path in tqdm(path_list):
    out_path = os.path.join(out_dir,path.split('/')[-1])
    file_exists = glob.glob(out_path)
    if(len(file_exists) != 0):
      print("File Already exists: " , out_path)
      continue
    frames = []
    flag = 0
    face_all = []
    frames1 = []
    out = cv2.VideoWriter(out_path,cv2.VideoWriter_fourcc('M','J','P','G'), 30, (112,112))
    for idx,frame in enumerate(frame_extract(path)):
      #if(idx % 3 == 0):
      if(idx <= 150):
        frames.append(frame)
        if(len(frames) == 4):
          faces = face_recognition.batch_face_locations(frames)
          for i,face in enumerate(faces):
            if(len(face) != 0):
              top,right,bottom,left = face[0]
            try:
              out.write(cv2.resize(frames[i][top:bottom,left:right,:],(112,112)))
            except:
              pass
          frames = []
    try:
      del top,right,bottom,left
    except:
      pass
    out.release()

mkdir: cannot create directory ‘/content/drive/My Drive/FaceData’: File exists


In [ ]:
create_face_videos(video_files,'/content/drive/My Drive/FaceData/')

No of videos already present  232


  0%|          | 0/1698 [00:00<?, ?it/s]

File Already exists:  /content/drive/My Drive/FaceData/qadqzqlgcf.mp4


KeyboardInterrupt: ignored

In [ ]:
def validate_video(vid_path,train_transforms):
      transform = train_transforms
      count = 20
      video_path = vid_path
      frames = []
      a = int(100/count)
      first_frame = np.random.randint(0,a)
      temp_video = video_path.split('/')[-1]
      for i,frame in enumerate(frame_extract(video_path)):
        frames.append(transform(frame))
        if(len(frames) == count):
          break
      frames = torch.stack(frames)
      frames = frames[:count]
      return frames
#extract a from from video
def frame_extract(path):
  vidObj = cv2.VideoCapture(path)
  success = 1
  while success:
      success, image = vidObj.read()
      if success:
          yield image

im_size = 112
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

train_transforms = transforms.Compose([
                                        transforms.ToPILImage(),
                                        transforms.Resize((im_size,im_size)),
                                        transforms.ToTensor(),
                                        transforms.Normalize(mean,std)])
video_fil =  glob.glob('/content/drive/My Drive/FaceData/*.mp4')
print("Total no of videos :" , len(video_fil))
print(video_fil)
count = 0;
for i in video_fil:
  try:
    count+=1
    validate_video(i,train_transforms)
  except:
    print("Number of video processed: " , count ," Remaining : " , (len(video_fil) - count))
    print("Corrupted video is : " , i)
    continue
print((len(video_fil) - count))

Total no of videos : 1468
['/content/drive/My Drive/RealDataset/yidrboessk.mp4', '/content/drive/My Drive/RealDataset/jwuyeomibc.mp4', '/content/drive/My Drive/RealDataset/eynxajczkd.mp4', '/content/drive/My Drive/RealDataset/itpauysdvo.mp4', '/content/drive/My Drive/RealDataset/pdzadxxidz.mp4', '/content/drive/My Drive/RealDataset/aunqfjmfqp.mp4', '/content/drive/My Drive/RealDataset/ortvcnoghh.mp4', '/content/drive/My Drive/RealDataset/wrsxxqkvay.mp4', '/content/drive/My Drive/RealDataset/uyvzsejsbq.mp4', '/content/drive/My Drive/RealDataset/eiayatksud.mp4', '/content/drive/My Drive/RealDataset/tfbjbnmyxo.mp4', '/content/drive/My Drive/RealDataset/tlsdqbpcps.mp4', '/content/drive/My Drive/RealDataset/tjhpwqxqlu.mp4', '/content/drive/My Drive/RealDataset/wvustidpql.mp4', '/content/drive/My Drive/RealDataset/zdkyyawcwe.mp4', '/content/drive/My Drive/RealDataset/rouhxkadcj.mp4', '/content/drive/My Drive/RealDataset/jsvqjkmpzu.mp4', '/content/drive/My Drive/RealDataset/prcobqjqfa.mp4', '

In [ ]:
#Metadatada durumu belirtilmiş verleri ayırma
# Csv dosyası yolun
csv_file_path = '/content/drive/MyDrive/metadata.csv'

# Hedef klasör yolu
target_folder_path = '/content/drive/MyDrive/RealDataset'

#Kaynak klasör yolu
source_folder_path = '/content/drive/MyDrive/FaceData'

# CSV dosyasını okuyun ve dosya adlarını bir liste olarak alın
file_names = []
with open(csv_file_path, 'r') as file:
    csv_reader = csv.reader(file)
    for row in csv_reader:
        file_names.extend(row)

# Dosyaları hedef klasöre taşıma
for file_name in file_names:
    source_file_path = os.path.join(source_folder_path, file_name)
    target_file_path = os.path.join(target_folder_path, file_name)
    try:
        shutil.move(source_file_path, target_file_path)
        print('Dosya taşındı: %s' % file_name)
    except FileNotFoundError:
        print('Dosya bulunamadı: %s' % file_name)
    except shutil.Error:
        print('Dosya taşıma hatası: %s' % file_name)

In [ ]:
# Video okuyucusunu başlatma
target_folder_path = '/content/drive/MyDrive/RealImageDataset'
source_folder_path = '/content/drive/MyDrive/RealDataset/*'
csv_file_path = '/content/drive/MyDrive/metadata.csv'
def video_to_frames(input_file, output_folder):
  cap2 = cv2.VideoCapture(source_folder_path)

  # Çerçeve sayacını başlatma
  frame_count = 0

  # Çerçevelerin kaydedileceği klasörü oluşturma
  os.makedirs(target_folder_path, exist_ok=True)

  while cap.isOpened():
     # Video çerçevesini okuma
    ret, frame = cap.read()

  if ret:
    # Çerçeve sayısını artırma
     frame_count += 1

     # Çerçeveyi kaydetme
     frame_path = os.path.join(output_folder, f"frame_{frame_count}.jpg")
     cv2.imwrite(frame_path, frame)

    # İşlenen çerçeveyi görüntüleme
     cv2.imshow('Frame', frame)

     # 'q' tuşuna basıldığında döngüden çıkma
  if cv2.waitKey(1) & 0xFF == ord('q'):
  else:
    break

  # Video okuyucusunu ve pencereyi kapatma
  cap.release()
  cv2.destroyAllWindows()

# Video dosyasının yolu
input_file = "video.mp4"

# Çerçevelerin kaydedileceği klasörün yolu
output_folder = "frames"

def process_videos_in_directory(input_directory, output_directory):
    # Girdi ve çıktı klasörlerini oluşturma
    os.makedirs(output_directory, exist_ok=True)

    # Girdi klasöründeki tüm dosya ve dizinleri listeleme
    files = os.listdir(input_directory)

    # Dosya ve dizinleri dolaşma
    for file in files:
        file_path = os.path.join(input_directory, file)

        # Dosya ise işlem yap
        if os.path.isfile(file_path):
            # Dosya uzantısını kontrol etme (sadece videoları işleme)
            _, file_extension = os.path.splitext(file_path)
            if file_extension in ['.mp4', '.avi', '.mkv']:
                # Video dosyasını çerçevelere dönüştürme
                video_to_frames(file_path, output_directory)

        # Dizin ise tekrar fonksiyonu çağırarak içeriğini işleme
        elif os.path.isdir(file_path):
            subdirectory_output = os.path.join(output_directory, file)
            process_videos_in_directory(file_path, subdirectory_output)

# Girdi ve çıktı klasörlerinin yolları
input_directory = "input_folder"
output_directory = "output_folder"

# Videoları çerçevelere dönüştürme işlemini başlatma
process_videos_in_directory(source_folder_path, target_folder_path)

IndentationError: ignored